# 1. Ingestione Dati e Preprocessing Spaziale
**Progetto:** La Topologia della Resilienza Urbana  
**Autore:** Urban Network Resilience Lab  
**Descrizione:** Questo notebook documenta l'intera pipeline di acquisizione dei dati grezzi (*Raw Data*) e la loro trasformazione in geometrie e attributi pronti per la modellazione in teoria dei grafi (*L-Space*). Viene mostrato come i dati fisici della città di Bologna vengono integrati spazialmente in un unico dataset coerente.

## 1.1 Fonti dei Dati e Origine

La modellazione della rete di trasporto di Bologna si basa sull'integrazione di quattro fonti informative eterogenee:

1. **Rete di Superficie Gomma (Bus):** Estratta dal feed GTFS ufficiale della città di Bologna (`gommagtfsbo`). I nodi sono definiti in `stops.txt` (fermate urbane), mentre gli archi topologici sono ricavati sequenzializzando `stop_times.txt` e filtrando i `trip_id` tramite una white-list rigida delle sole linee urbane (`11`, `13`, `14`, `32`, `33`, ecc.) per escludere il rumore extraurbano e suburbano a lunga distanza.
2. **Rete Tramviaria Futura (Linea Rossa e Verde):** Modellata partendo dalle planimetrie ufficiali del Comune di Bologna (progetto PUMS 2030), tradotte in coordinate geografiche JSON (`tram_data`).
3. **Flussi di Traffico Veicolare (Open Data Comune):** Dataset delle spire di rilevamento comunali (`bologna_spire_traffic.csv`). Ciascun sensore registra i transiti orari su 24 colonne (`00_00_01_00` fino a `23_00_24_00`).
4. **Sinistrosità Storica (Open Data Comune):** Dataset degli incidenti stradali registrati sul territorio comunale (`bologna_accidents.csv`), georeferenziati tramite stringhe di punti `geo_point_2d`.

## 1.2 Metodologia e Modello Scientifico

Per stimare i livelli di congestione e il rischio ambientale associato alle fermate del trasporto pubblico, vengono applicati i seguenti modelli matematici:

* **Estrazione del Picco Orario (Ora di Punta):** Per calcolare la congestione reale nei pressi di ciascun sensore stradale, viene estratto il picco massimo orario registrato da ciascuna spira in una giornata tipo:
  $$V_{picco} = \max_{h \in [1, 24]} (\text{Flusso}_h)$$
  
* **Stima della Capacità Empirica (Empirical Capacity Estimation):** A causa dell'assenza di dati precisi sulla larghezza e sul numero di corsie di ogni vicolo del centro storico, la capacità nominale oraria $C_{via}$ viene estratta analizzando il massimo storico assoluto registrato da quel sensore, moltiplicato per un coefficiente di tolleranza ingegneristica ($1.1$):
  $$C_{via} = \max (\text{Picco Orario Giorno}) \times 1.1$$
  Viene imposto un limite inferiore rigido ($C_{min} = 300 \text{ veicoli/ora}$) per salvaguardare la computazione ed evitare divisioni per zero nei vicoli storici più stretti.
  
* **Associazione Spaziale (GIS Snapping):** Tramite la libreria `geopandas`, le fermate del bus vengono convertite in un GeoDataFrame proiettato in coordinate metriche (`EPSG:32632`). Viene eseguito un *Spatial Join Nearest* (`gpd.sjoin_nearest`) per catturare il flusso di picco e la capacità della spira automobilistica più vicina (entro un raggio di tolleranza). Gli incidenti vengono invece aggregati creando un buffer circolare di raggio $300\text{m}$ attorno ad ogni fermata e contando le intersezioni geometriche reali (`gpd.sjoin` con predicato `intersects`).

In [3]:
import pandas as pd
from pathlib import Path
import sys

# Aggiungiamo la cartella di root del progetto al path per importare src
sys.path.append(str(Path("..").resolve()))

from src.preprocessing.bus import process_bus_network

print("--- Step 1: Esecuzione della Pipeline di Preprocessing ---")
# Esegue il parsing del GTFS, lo spatial join delle spire e il calcolo dei buffer degli incidenti
process_bus_network()

--- Step 1: Esecuzione della Pipeline di Preprocessing ---
Inizio Preprocessing Rete Bus (Hard Filter Linee Bologna)...
 -> Analisi Linee (Filtraggio tramite White-List)...
 -> Estrazione Archi (Costruzione topologia pura)...
 -> Elaborazione Nodi (Rimozione orfani extraurbani)...
 -> Mappatura Spire di Traffico e Incidenti...
✅ Pipeline Completata con Successo (White-List)!
   - Nodi (Fermate Bologna Urbana): 1253
   - Archi (Connessioni Topologiche): 2099


In [4]:
# Visualizzazione dei dati pre-elaborati esportati in bologna_stations.csv
df_stations = pd.read_csv(Path("../dataset/bologna/processed/bologna_stations.csv"))
print(f"\nStazioni Urbane Validate con Attributi Geografici ed Ambientali: {len(df_stations)}")
df_stations.head()


Stazioni Urbane Validate con Attributi Geografici ed Ambientali: 1253


,stop_id,stop_name,stop_lat,stop_lon,nearest_traffic_flow,road_capacity,accidents_300m
0,1,STAZIONE CENTRALE,44.505766,11.343176,199.28,1224.3,0
1,10,PORTA GALLIERA,44.504443,11.346387,298.17,1652.2,0
2,100,PIAZZA MAGGIORE,44.494369,11.343672,0.00,300.0,0
3,10001,IDICE,44.459853,11.436653,639.32,910.8,0
4,10002,IDICE,44.460035,11.436539,639.32,910.8,0


## 1.3 Considerazioni Ingegneristiche e Urbanistiche

Il preprocessing spaziale evidenzia chiaramente l'eterogeneità strutturale della città di Bologna:
* Le fermate ubicate all'interno del nucleo medievale (es. *Via Farini*, *Via San Vitale*, *Via Santo Stefano*) ereditano dai sensori di traffico limitrofi una capacità empirica molto bassa ($C \approx 300 - 450\text{ veicoli/ora}$). Questo rispecchia le sezioni stradali ridotte e le limitazioni fisiche della viabilità storica.
* Le stazioni poste lungo i *Viali di Circonvallazione* o sulle grandi radiali di penetrazione urbana (es. *Via Stalingrado*, *Via Mazzini*) ereditano flussi e capacità significativamente superiori ($C > 1500\text{ veicoli/ora}$).

Questo ancoraggio spaziale dei dati reali impedisce al modello di assumere una fluidità omogenea della città, permettendo al simulatore di catturare le reali strozzature del tessuto medievale durante le fasi successive di congestione non lineare.